In [ ]:
# ============================================
# 1. Introduction
# ============================================

# This notebook builds the customer behavior funnel
# using CustomerStatus, ChurnCategory, and segments (Cluster).
# It includes:
# - Funnel stage definition
# - Conversion rate calculation
# - Funnel visualization
# - Comparison by segment

# ============================================
# 2. Load libraries
# ============================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# ============================================
# 3. Load segmented dataset
# ============================================

df = pd.read_csv("../data/processed/segments_telco.csv")
df.head()

funnel_order = ["Joined", "Stayed", "Churned"]

df["CustomerStatus"] = df["CustomerStatus"].astype("category")
df["CustomerStatus"] = df["CustomerStatus"].cat.set_categories(funnel_order)

funnel_counts = df["CustomerStatus"].value_counts()
funnel_counts

plt.figure(figsize=(8,6))
sns.barplot(
    x=funnel_counts.values,
    y=funnel_counts.index,
    palette="viridis"
)
plt.title("CustomerStatus Funnel")
plt.xlabel("Number of customers")
plt.ylabel("Stage")
plt.show()

joined = funnel_counts["Joined"]
stayed = funnel_counts["Stayed"]
churned = funnel_counts["Churned"]

conversion_stayed = stayed / joined
conversion_churn = churned / stayed

conversion_stayed, conversion_churn

segment_funnel = df.groupby(["Cluster", "CustomerStatus"]).size().unstack().fillna(0)
segment_funnel

plt.figure(figsize=(12,6))
sns.heatmap(segment_funnel, annot=True, cmap="magma")
plt.title("Funnel by Segment (Cluster)")
plt.show()

segment_churn_rate = (
    segment_funnel["Churned"] /
    (segment_funnel["Joined"] + segment_funnel["Stayed"])
)

segment_churn_rate

plt.figure(figsize=(10,4))
sns.barplot(
    x=segment_churn_rate.index,
    y=segment_churn_rate.values,
    palette="coolwarm"
)
plt.title("Churn Rate by Segment")
plt.ylabel("Churn Rate")
plt.show()

churn_reasons = (
    df[df["CustomerStatus"] == "Churned"]["ChurnReason"]
    .value_counts()
    .head(10)
)

churn_reasons

plt.figure(figsize=(10,6))
sns.barplot(
    x=churn_reasons.values,
    y=churn_reasons.index,
    palette="inferno"
)
plt.title("Top 10 Churn Reasons")
plt.xlabel("Number of customers")
plt.show()

reason_segment = (
    df[df["CustomerStatus"] == "Churned"]
    .groupby(["Cluster", "ChurnReason"])
    .size()
    .unstack()
    .fillna(0)
)

reason_segment.head()

df.to_csv("../data/processed/funnel_telco.csv", index=False)
print("Funnel saved to data/processed/funnel_telco.csv")

print("""
FUNNEL CONCLUSIONS:

1. CustomerStatus allows building a clear funnel: Joined -> Stayed -> Churned.
2. Conversion rates between stages were calculated.
3. Segments with higher churn were identified.
4. The main churn reasons were analyzed.
5. A per-segment funnel was generated for actionable insights.
6. The dataset is ready for modeling in 06_Churn_Model.ipynb.
""")
